# This is model comparison.

## OBjectives:

This is only for a discussion (Not meant to be a solution). I am not sure if this correct approach or not yet. I am still learning and investigating. 

- Implement a NN model and compare the accuracy with KNN and XGBoost.
- Create a weak condition test scenarios and compare the results among the baselines and NN model.

## NOTE:

This is not my code. I highly leverage the following codes. 

https://github.com/homerjed/set_transformer
https://github.com/juho-lee/set_transformer 
https://github.com/Enigmatisms/Set-Transformer 

If we decided to use this model, we have to refactor and make it our own code for the project.

## Data Preparation

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

# Assumes you already have:
# train = TrainingData.csv (19937 rows)
# val   = ValidationData.csv (1111 rows)
# If not, uncomment and set paths:
train = pd.read_csv("data/UjiIndoorLoc/TrainingData.csv")
val   = pd.read_csv("data/UjiIndoorLoc/ValidationData.csv")

wap_cols = [c for c in train.columns if c.upper().startswith("WAP")]
assert len(wap_cols) == 520, f"Expected 520 WAP cols, got {len(wap_cols)}"

# Raw UJI encoding: missing = 100, detected RSSI is negative
X_train_uji = train[wap_cols].to_numpy(dtype=np.float32).copy()
X_val_uji   = val[wap_cols].to_numpy(dtype=np.float32).copy()

# Targets: use dataset coordinates as XY (projected, meter-like). Center for numerical stability.
y_train_xy = train[["LONGITUDE", "LATITUDE"]].to_numpy(dtype=np.float32)
y_val_xy   = val[["LONGITUDE", "LATITUDE"]].to_numpy(dtype=np.float32)
origin = y_train_xy.mean(axis=0, keepdims=True)
y_train_xy_c = y_train_xy - origin
y_val_xy_c   = y_val_xy - origin

# Tabular features for KNN/XGB: replace missing 100 -> -110
X_train_tab = X_train_uji.copy()
X_val_tab   = X_val_uji.copy()
X_train_tab[X_train_tab == 100.0] = -110.0
X_val_tab[X_val_tab == 100.0]     = -110.0

# Standardize for KNN/XGB (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_tab)
X_val_scaled   = scaler.transform(X_val_tab)

print("Shapes:")
print("X_train_uji:", X_train_uji.shape, "X_val_uji:", X_val_uji.shape)
print("X_train_scaled:", X_train_scaled.shape, "X_val_scaled:", X_val_scaled.shape)
print("y_train_xy_c:", y_train_xy_c.shape, "y_val_xy_c:", y_val_xy_c.shape)

print("Detected APs (median): train =", int(np.median((X_train_uji != 100).sum(axis=1))),
      "| val =", int(np.median((X_val_uji != 100).sum(axis=1))))

Shapes:
X_train_uji: (19937, 520) X_val_uji: (1111, 520)
X_train_scaled: (19937, 520) X_val_scaled: (1111, 520)
y_train_xy_c: (19937, 2) y_val_xy_c: (1111, 2)
Detected APs (median): train = 17 | val = 15


## Train baselines (KNN + XGBoost regression) for distance accuracy

In [3]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

def dist_stats(y_true, y_pred):
    d = np.linalg.norm(y_true - y_pred, axis=1)
    return {
        "mean_m": float(np.mean(d)),
        "median_m": float(np.median(d)),
        "rmse_m": float(np.sqrt(np.mean(d**2))),
        "p90_m": float(np.percentile(d, 90)),
    }

# ---- KNN regression
knn_reg = KNeighborsRegressor(n_neighbors=5, weights="distance", metric="euclidean", n_jobs=-1)
knn_reg.fit(X_train_scaled, y_train_xy_c)
pred_knn_clean = knn_reg.predict(X_val_scaled).astype(np.float32)
print("[KNN clean]", dist_stats(y_val_xy_c, pred_knn_clean))

# ---- XGBoost regression (multi-output)
xgb_base = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)
xgb_reg = MultiOutputRegressor(xgb_base)
xgb_reg.fit(X_train_scaled, y_train_xy_c)
pred_xgb_clean = xgb_reg.predict(X_val_scaled).astype(np.float32)
print("[XGB clean]", dist_stats(y_val_xy_c, pred_xgb_clean))

[KNN clean] {'mean_m': 12.548238754272461, 'median_m': 7.3377366065979, 'rmse_m': 23.547895431518555, 'p90_m': 26.982656478881836}
[XGB clean] {'mean_m': 19.523157119750977, 'median_m': 12.573505401611328, 'rmse_m': 28.60770034790039, 'p90_m': 42.8552360534668}


##  Corruption function (applied ONCE to the val set, then shared by all models)

In [4]:
def apply_corruptions_uji(X_uji, seed, dropout_rate=0.0, bias_db=0.0, noise_std=0.0, missing_value=100.0):
    """
    X_uji: float32, missing=100, detected are negative RSSI values.
    Corrupt detected entries only:
      - bias_db: add constant offset to detected RSSI
      - noise_std: add Gaussian noise to detected RSSI
      - dropout_rate: randomly set detected RSSI -> 100 (missing)
    """
    rng = np.random.default_rng(seed)
    Xc = X_uji.astype(np.float32).copy()
    detected = Xc != missing_value

    if bias_db != 0.0:
        Xc[detected] = Xc[detected] + float(bias_db)

    if noise_std > 0.0:
        noise = rng.normal(0.0, float(noise_std), size=Xc.shape).astype(np.float32)
        Xc[detected] = Xc[detected] + noise[detected]

    # clamp detected RSSI
    Xc[detected] = np.clip(Xc[detected], -110.0, -30.0)

    if dropout_rate > 0.0:
        drop_mask = rng.random(Xc.shape) < float(dropout_rate)
        Xc[detected & drop_mask] = missing_value

    return Xc

def prep_tabular_and_scale(X_uji_corrupted):
    X_tab = X_uji_corrupted.copy()
    X_tab[X_tab == 100.0] = -110.0
    X_scaled = scaler.transform(X_tab)  # scaler fitted on clean train
    return X_scaled.astype(np.float32)

In [5]:
# =========================
# Shared test setup (ONE cell) for ALL models
# - Builds the exact same clean + corrupted validation/test sets once
# - KNN/XGB use: TEST_CASES[name]['X_scaled']
# - Set-Transformer uses: TEST_CASES[name]['X_uji']
# =========================
from collections import OrderedDict
import numpy as np

# Deterministic corruption seed base
MASTER_SEED = 123

# Main corruption settings you want to report
settings = [
    ("clean", 0.0, 0.0, 0.0),
    ("dropout_0.30", 0.30, 0.0, 0.0),
    ("dropout_0.50", 0.50, 0.0, 0.0),
    ("drop0.40_bias5_noise3", 0.40, 5.0, 3.0),
]

TEST_CASES = OrderedDict()

for i, (name, dr, bd, ns) in enumerate(settings):
    if name == "clean":
        X_uji = X_val_uji
    else:
        X_uji = apply_corruptions_uji(
            X_val_uji,
            seed=MASTER_SEED + i,
            dropout_rate=dr,
            bias_db=bd,
            noise_std=ns
        )

    X_scaled = prep_tabular_and_scale(X_uji)
    median_detected = int(np.median((X_uji != 100).sum(axis=1)))

    TEST_CASES[name] = {
        "X_uji": X_uji,
        "X_scaled": X_scaled,
        "median_detected": median_detected,
        "params": {"dropout_rate": dr, "bias_db": bd, "noise_std": ns},
    }

print("Built TEST_CASES:")
for name, case in TEST_CASES.items():
    p = case["params"]
    print(f"  - {name}: median_detected={case['median_detected']} | "
          f"dropout={p['dropout_rate']}, bias={p['bias_db']}, noise={p['noise_std']}")


Built TEST_CASES:
  - clean: median_detected=15 | dropout=0.0, bias=0.0, noise=0.0
  - dropout_0.30: median_detected=10 | dropout=0.3, bias=0.0, noise=0.0
  - dropout_0.50: median_detected=8 | dropout=0.5, bias=0.0, noise=0.0
  - drop0.40_bias5_noise3: median_detected=9 | dropout=0.4, bias=5.0, noise=3.0


## Cell 4 — Run the same corrupted test across KNN + XGBoost (sweep)

In [6]:
def eval_knn_xgb_on_case(case, label=""):
    X_val_scaled_cor = case["X_scaled"]

    pred_knn = knn_reg.predict(X_val_scaled_cor).astype(np.float32)
    pred_xgb = xgb_reg.predict(X_val_scaled_cor).astype(np.float32)

    out = {
        "setting": label,
        "knn": dist_stats(y_val_xy_c, pred_knn),
        "xgb": dist_stats(y_val_xy_c, pred_xgb),
        "detected_median": case["median_detected"],
    }
    return out

results_baselines = []
for name, case in TEST_CASES.items():
    r = eval_knn_xgb_on_case(case, label=name)
    results_baselines.append(r)

for r in results_baselines:
    print(f"\n== {r['setting']} ==  (median detected APs: {r['detected_median']} )")
    print("KNN:", r["knn"])
    print("XGB:", r["xgb"])



== clean ==  (median detected APs: 15 )
KNN: {'mean_m': 12.548238754272461, 'median_m': 7.3377366065979, 'rmse_m': 23.547895431518555, 'p90_m': 26.982656478881836}
XGB: {'mean_m': 19.523157119750977, 'median_m': 12.573505401611328, 'rmse_m': 28.60770034790039, 'p90_m': 42.8552360534668}

== dropout_0.30 ==  (median detected APs: 10 )
KNN: {'mean_m': 24.01778793334961, 'median_m': 12.054169654846191, 'rmse_m': 42.9449348449707, 'p90_m': 55.04472732543945}
XGB: {'mean_m': 35.976802825927734, 'median_m': 25.887041091918945, 'rmse_m': 47.26510238647461, 'p90_m': 80.43913269042969}

== dropout_0.50 ==  (median detected APs: 8 )
KNN: {'mean_m': 45.7103271484375, 'median_m': 20.62412452697754, 'rmse_m': 71.80299377441406, 'p90_m': 152.08056640625}
XGB: {'mean_m': 54.31508255004883, 'median_m': 44.111305236816406, 'rmse_m': 67.75273895263672, 'p90_m': 115.96366882324219}

== drop0.40_bias5_noise3 ==  (median detected APs: 9 )
KNN: {'mean_m': 26.694746017456055, 'median_m': 12.829230308532715,

## Cell 5 — Set-Transformer (Gaussian two-head) model + training (train once, test many corruptions)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

def normalize_rssi(rssi):
    r = rssi.astype(np.float32)
    r = np.clip(r, -110.0, -30.0)
    return 2.0 * (r - (-110.0)) / 80.0 - 1.0  # [-110,-30] -> [-1,1]

def build_tokens_with_cls(X_uji, k_tokens=64, missing_value=100.0):
    """
    Returns tokens for Set-Transformer:
      ap_ids: (N, 1+K) int64
      rssi_vals: (N, 1+K, 1) float32 normalized
      pad_mask: (N, 1+K) bool, True=masked
    CLS token prevents NaNs when a row has 0 detected APs.
    """
    N, D = X_uji.shape
    cls_id = D  # embedding size must be D+1

    ap_ids = np.full((N, k_tokens + 1), cls_id, dtype=np.int64)
    rssi_vals = np.zeros((N, k_tokens + 1, 1), dtype=np.float32)
    pad_mask = np.ones((N, k_tokens + 1), dtype=bool)
    pad_mask[:, 0] = False  # CLS always present

    for i in range(N):
        row = X_uji[i]
        det = np.where(row != missing_value)[0]
        if det.size == 0:
            continue
        det_sorted = det[np.argsort(row[det])[::-1]]
        chosen = det_sorted[:k_tokens]
        ap_ids[i, 1:1+chosen.size] = chosen.astype(np.int64)
        rssi_vals[i, 1:1+chosen.size, 0] = normalize_rssi(row[chosen])
        pad_mask[i, 1:1+chosen.size] = False

    return ap_ids, rssi_vals, pad_mask

class TokenDataset(Dataset):
    def __init__(self, ap_ids, rssi_vals, pad_mask, y_xy):
        self.ap_ids = torch.from_numpy(ap_ids).long()
        self.rssi_vals = torch.from_numpy(rssi_vals).float()
        self.pad_mask = torch.from_numpy(pad_mask).bool()
        self.y = torch.from_numpy(y_xy).float()

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.ap_ids[idx], self.rssi_vals[idx], self.pad_mask[idx], self.y[idx]

class SetTransformerGaussian(nn.Module):
    def __init__(self, num_tokens, d_model=64, n_heads=4, n_layers=3, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Embedding(num_tokens, d_model)
        self.rssi_proj = nn.Sequential(nn.Linear(1, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.mu_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, 2))
        self.sigma_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, 2))

    def forward(self, ap_ids, rssi_vals, pad_mask):
        tok = self.token_embed(ap_ids) + self.rssi_proj(rssi_vals)
        h = self.encoder(tok, src_key_padding_mask=pad_mask)
        g = h[:, 0, :]  # CLS pooling
        mu = self.mu_head(g)
        sigma = F.softplus(self.sigma_head(g)) + 1e-3
        sigma = torch.clamp(sigma, 1e-3, 1e3)
        return mu, sigma

def gaussian_nll(y, mu, sigma):
    z = (y - mu) / sigma
    return (0.5*(z**2) + torch.log(sigma)).sum(dim=1).mean()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Train/val split from TRAINING set only
idx = np.arange(X_train_uji.shape[0])
idx_tr, idx_va = train_test_split(idx, test_size=0.15, random_state=42, shuffle=True)

X_tr_uji, y_tr = X_train_uji[idx_tr], y_train_xy_c[idx_tr]
X_va_uji, y_va = X_train_uji[idx_va], y_train_xy_c[idx_va]

# Tokenize
K_TOKENS = 64
ap_tr, rssi_tr, mask_tr = build_tokens_with_cls(X_tr_uji, k_tokens=K_TOKENS)
ap_va, rssi_va, mask_va = build_tokens_with_cls(X_va_uji, k_tokens=K_TOKENS)

ds_tr = TokenDataset(ap_tr, rssi_tr, mask_tr, y_tr)
ds_va = TokenDataset(ap_va, rssi_va, mask_va, y_va)

train_loader = DataLoader(ds_tr, batch_size=256, shuffle=True)
val_loader   = DataLoader(ds_va, batch_size=256, shuffle=False)

# Model
num_tokens = X_train_uji.shape[1] + 1  # 520 + CLS
st_model = SetTransformerGaussian(num_tokens=num_tokens, d_model=64, n_heads=4, n_layers=3, dropout=0.1).to(device)
opt = torch.optim.AdamW(st_model.parameters(), lr=3e-4, weight_decay=1e-3)

# Train a few epochs (CPU-friendly)
best_rmse = float("inf")
best_state = None

def eval_loader(model, loader):
    model.eval()
    mus, sigmas, ys = [], [], []
    with torch.no_grad():
        for ap_ids, rssi_vals, pad_mask, y in loader:
            ap_ids = ap_ids.to(device)
            rssi_vals = rssi_vals.to(device)
            pad_mask = pad_mask.to(device)
            y = y.to(device)
            mu, sigma = model(ap_ids, rssi_vals, pad_mask)
            mus.append(mu.cpu().numpy())
            sigmas.append(sigma.cpu().numpy())
            ys.append(y.cpu().numpy())
    mu_np = np.concatenate(mus, axis=0)
    sigma_np = np.concatenate(sigmas, axis=0)
    y_np = np.concatenate(ys, axis=0)
    d = np.linalg.norm(y_np - mu_np, axis=1)
    return dist_stats(y_np, mu_np), mu_np, sigma_np, y_np, d

EPOCHS = 20
for ep in range(1, EPOCHS+1):
    st_model.train()
    total = 0.0
    for ap_ids, rssi_vals, pad_mask, y in train_loader:
        ap_ids = ap_ids.to(device)
        rssi_vals = rssi_vals.to(device)
        pad_mask = pad_mask.to(device)
        y = y.to(device)

        mu, sigma = st_model(ap_ids, rssi_vals, pad_mask)
        loss = gaussian_nll(y, mu, sigma)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(st_model.parameters(), 1.0)
        opt.step()
        total += float(loss.item())

    val_stats, *_ = eval_loader(st_model, val_loader)
    print(f"ep {ep:02d} train_nll={total/len(train_loader):.4f} | val_rmse={val_stats['rmse_m']:.3f}")

    if val_stats["rmse_m"] < best_rmse:
        best_rmse = val_stats["rmse_m"]
        best_state = {k: v.detach().cpu().clone() for k, v in st_model.state_dict().items()}

st_model.load_state_dict(best_state)
print("best val rmse:", best_rmse)

device: cpu


/home/iptracej/miniforge3/envs/uji-indoorloc/lib/python3.11/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


ep 01 train_nll=3671.9698 | val_rmse=141.667
ep 02 train_nll=366.6317 | val_rmse=141.637
ep 03 train_nll=94.4923 | val_rmse=141.549
ep 04 train_nll=37.1718 | val_rmse=140.254
ep 05 train_nll=19.2097 | val_rmse=120.753
ep 06 train_nll=11.5393 | val_rmse=81.791
ep 07 train_nll=8.4666 | val_rmse=52.917
ep 08 train_nll=7.5447 | val_rmse=39.608
ep 09 train_nll=6.9153 | val_rmse=35.233
ep 10 train_nll=6.5518 | val_rmse=32.269
ep 11 train_nll=6.3273 | val_rmse=29.145
ep 12 train_nll=6.1088 | val_rmse=25.092
ep 13 train_nll=5.8927 | val_rmse=21.619
ep 14 train_nll=5.7126 | val_rmse=18.531
ep 15 train_nll=5.5035 | val_rmse=15.830
ep 16 train_nll=5.3240 | val_rmse=14.412
ep 17 train_nll=5.1661 | val_rmse=13.379
ep 18 train_nll=5.0331 | val_rmse=13.241
ep 19 train_nll=4.9419 | val_rmse=11.988
ep 20 train_nll=4.8707 | val_rmse=11.787
best val rmse: 11.786831855773926


## Set-Transformer NORMAL baseline (clean test)

In [8]:
# =========================
# Cell A: Set-Transformer NORMAL baseline (no corruption)
# =========================
import numpy as np
import torch
from torch.utils.data import DataLoader

def eval_set_transformer_metrics_on_uji(X_uji_test, y_true_xy_c, batch_size=256):
    """
    Returns:
      st_stats: dict with mean/median/rmse/p90 (same as KNN/XGB)
      unc_err_corr: correlation between ||sigma|| and distance error
      cov95: empirical coverage for 95% ellipse under diagonal Gaussian
    """
    ap_te, rssi_te, mask_te = build_tokens_with_cls(X_uji_test, k_tokens=K_TOKENS)
    ds_te = TokenDataset(ap_te, rssi_te, mask_te, y_true_xy_c)
    loader = DataLoader(ds_te, batch_size=batch_size, shuffle=False)

    # eval_loader returns: (stats, mu_np, sigma_np, y_np, d)
    st_stats, mu_np, sigma_np, y_np, d = eval_loader(st_model, loader)

    # uncertainty-error correlation
    unc = np.linalg.norm(sigma_np, axis=1)
    unc_err_corr = float(np.corrcoef(unc, d)[0, 1]) if np.isfinite(unc).all() else float("nan")

    # 95% coverage for diagonal Gaussian ellipse
    # md2 = (dx/sx)^2 + (dy/sy)^2 <= q, q = chi2.ppf(0.95, df=2) = -2 ln(1-0.95)
    q = -2.0 * np.log(1.0 - 0.95)
    z = (y_np - mu_np) / sigma_np
    md2 = (z ** 2).sum(axis=1)
    cov95 = float(np.mean(md2 <= q))

    return st_stats, unc_err_corr, cov95

# Evaluate on CLEAN validation set (no corruption)
st_stats_clean, st_corr_clean, st_cov95_clean = eval_set_transformer_metrics_on_uji(TEST_CASES['clean']['X_uji'], y_val_xy_c)

median_detected_clean = TEST_CASES['clean']['median_detected']

print(f"\n== clean ==  (median detected APs: {median_detected_clean} )")
print("ST :", st_stats_clean)
print("   uncertainty-error corr:", round(st_corr_clean, 3), "| coverage@95%:", round(st_cov95_clean, 3))


== clean ==  (median detected APs: 15 )
ST : {'mean_m': 10.647356986999512, 'median_m': 8.37416934967041, 'rmse_m': 13.889679908752441, 'p90_m': 19.578784942626953}
   uncertainty-error corr: 0.373 | coverage@95%: 0.854


## Set-Transformer CORRUPTION tests (same conditions as KNN/XGB)

In [9]:
# =========================
# Cell B: Set-Transformer CORRUPTION tests (same settings as KNN/XGB)
# =========================
print("\n=== Set-Transformer CORRUPTION tests ===")

for name, case in TEST_CASES.items():
    if name == "clean":
        continue  # already done in Cell A

    X_cor = case["X_uji"]
    median_detected = case["median_detected"]

    st_stats, st_corr, st_cov95 = eval_set_transformer_metrics_on_uji(X_cor, y_val_xy_c)

    print(f"\n== {name} ==  (median detected APs: {median_detected} )")
    print("ST :", st_stats)
    print("   uncertainty-error corr:", round(st_corr, 3), "| coverage@95%:", round(st_cov95, 3))



=== Set-Transformer CORRUPTION tests ===

== dropout_0.30 ==  (median detected APs: 10 )
ST : {'mean_m': 15.771702766418457, 'median_m': 12.21288013458252, 'rmse_m': 21.591304779052734, 'p90_m': 30.82347297668457}
   uncertainty-error corr: 0.468 | coverage@95%: 0.742

== dropout_0.50 ==  (median detected APs: 8 )
ST : {'mean_m': 23.024158477783203, 'median_m': 16.208850860595703, 'rmse_m': 33.905784606933594, 'p90_m': 46.61682891845703}
   uncertainty-error corr: 0.527 | coverage@95%: 0.635

== drop0.40_bias5_noise3 ==  (median detected APs: 9 )
ST : {'mean_m': 18.217439651489258, 'median_m': 13.56674575805664, 'rmse_m': 27.154664993286133, 'p90_m': 34.54446029663086}
   uncertainty-error corr: 0.475 | coverage@95%: 0.67


### Observation - Baselines degrade hard as signals get worse (expected)

- Clean
    - KNN RMSE 23.55, p90 26.98
    - XGB RMSE 28.61, p90 42.86

- Dropout 0.30

    - KNN RMSE 42.94, p90 55.04
    - XGB RMSE 47.27, p90 80.44

- Dropout 0.50

    - KNN RMSE 71.80, p90 152.08
    - XGB RMSE 67.75, p90 115.96

- Dropout 0.40 + bias5 + noise3

    - KNN RMSE 48.87, p90 63.24
    - XGB RMSE 55.05, p90 91.89

Both models gets worse when weak data condition progresses.

### A Neural Network (Set-Transformer) is clearly more robust in the “weak signal” settings

- Clean 
    - ST rmse_m': 13.889679908752441, 'p90_m': 19.578784942626953
    - This is better than KNN and XGB.

- Dropout 0.30
    - ST RMSE 21.59, p90 30.82
    - This is far better than KNN/XGB at the same condition.

- Dropout 0.50
    - ST RMSE 33.91, p90 46.62
    - Still dramatically better than KNN/XGB, which explode to 67–72 RMSE and triple-digit p90.

- Dropout 0.40 + bias5 + noise3
    - ST RMSE 27.15, p90 34.54
    - Again much better than KNN/XGB.

So the NN has better robustness, but still degrade slowly. This is something we should chase more.

